# Whisp COG Export Demo

This notebook demonstrates exporting Whisp multiband images to Google Cloud Storage as Cloud Optimized GeoTIFFs (COGs).

**Key features:**
- Uses GAUL raster for country masking (avoids memory issues with vector `clipToCollection`)
- Configurable band selection
- Configurable resolution (1000m for testing, 10m for production)

**Prerequisites:**
- Google Earth Engine authentication
- Access to GCS bucket (default: `whisp_bucket`)

In [ ]:
# Authenticate and initialize Earth Engine
import ee
ee.Authenticate()
ee.Initialize()

In [ ]:
# Import Whisp COG export functions
from openforis_whisp.export_cog import (
    export_whisp_image_to_cog,
    get_risk_bands,
    get_bands_by_theme,
    get_all_band_names,
    check_export_status,
    list_active_exports,
)

## 1. Explore Available Bands

Before exporting, let's see what bands are available.

In [ ]:
# Get bands used for risk assessment
risk_bands = get_risk_bands()
print(f"Risk bands ({len(risk_bands)} total):")
for band in risk_bands[:10]:
    print(f"  - {band}")
print("  ...")

In [ ]:
# Get bands by theme
treecover_bands = get_bands_by_theme(["treecover"])
print(f"\nTreecover bands ({len(treecover_bands)}):")
for band in treecover_bands:
    print(f"  - {band}")

disturbance_after_bands = get_bands_by_theme(["disturbance_after"])
print(f"\nDisturbance after 2020 bands ({len(disturbance_after_bands)}):")
for band in disturbance_after_bands[:10]:
    print(f"  - {band}")
if len(disturbance_after_bands) > 10:
    print("  ...")

## 2. Export COG for Côte d'Ivoire (Test at 1000m)

First test with a smaller resolution to verify the export works.

In [ ]:
# Export all bands for Côte d'Ivoire at 1000m (testing)
task_ci = export_whisp_image_to_cog(
    iso2_codes=["CI"],
    bucket="whisp_bucket",
    scale=1000,  # 1000m for testing
    cog_folder="whisp_cogs/test",
)

print(f"Task created: {task_ci.status()['description']}")
print("Starting export...")
task_ci.start()

In [ ]:
# Check export status
check_export_status(task_ci)

## 3. Export Risk Bands Only for Brazil (Test at 1000m)

For large countries like Brazil, exporting only risk-relevant bands reduces file size.

In [ ]:
# Export only risk-relevant bands for Brazil at 1000m
task_br = export_whisp_image_to_cog(
    iso2_codes=["BR"],
    bucket="whisp_bucket",
    bands=get_risk_bands(),  # Only risk bands
    scale=1000,
    cog_folder="whisp_cogs/test",
)

print(f"Task created: {task_br.status()['description']}")
print("Starting export...")
task_br.start()

In [ ]:
# Check export status
check_export_status(task_br)

## 4. Export with National Datasets (Côte d'Ivoire + Cocoa)

Include national cocoa dataset for Côte d'Ivoire.

In [ ]:
# Export with national cocoa dataset for Côte d'Ivoire
task_ci_national = export_whisp_image_to_cog(
    iso2_codes=["CI"],
    bucket="whisp_bucket",
    scale=1000,
    national_codes=["CI"],  # Include CI national datasets
    cog_folder="whisp_cogs/test",
    description="whisp_cog_CI_with_national",
)

print(f"Task created: {task_ci_national.status()['description']}")
print("Starting export...")
task_ci_national.start()

## 5. Monitor All Active Exports

In [ ]:
# List all active export tasks
active_tasks = list_active_exports()

## 6. Verify COG in GCS (after export completes)

Once the export is complete, verify the COG exists and inspect it.

In [ ]:
# Run this cell after export completes
# List files in the GCS bucket (requires gsutil or gcloud SDK)
!gsutil ls gs://whisp_bucket/whisp_cogs/test/

In [ ]:
# Inspect COG metadata using gdalinfo (via /vsicurl for remote access)
# Replace with actual filename from the export
!gdalinfo /vsigs/whisp_bucket/whisp_cogs/test/CI_1000m_20260213.tif

## Future Enhancements

Potential improvements for production use:

1. **Split exports by update frequency:**
   - Stable layers (annual): Export once, update yearly
   - Dynamic layers (weekly): Export more frequently

2. **Integration with local_stats.py:**
   - Add COG-based alternative path that reads from GCS instead of `getDownloadURL()`

3. **Scheduled refresh:**
   - Set up Cloud Scheduler to refresh dynamic layers automatically